# 🚀 Train Your First Custom Wake Word with Nanowakeword!

Welcome to the official tutorial for **Nanowakeword**!

In this notebook, we will guide you through the entire process of training a high-performance, custom wake word model from scratch. You don't need any pre-existing data—we will download everything we need and let Nanowakeword's intelligent engine do the heavy lifting.

**Our goal:** Go from zero to a ready-to-use wake word model in just a few simple steps. Let's get started!

**Installation**

In [7]:
# @title Žingsnis 1: Diegimas ir foninio triukšmo paruošimas
!pip install "nanowakeword[train] @ git+https://github.com/arcosoph/nanowakeword.git"
!pip install piper-tts

import shutil
import os, requests
from tqdm import tqdm
from datasets import load_dataset, Audio
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor

print("\nDiegimas baigtas. Siunčiami foniniai garsai...")

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# 1. RIR (Aidų) duomenys
rir_dir = "data/rir"
os.makedirs(rir_dir, exist_ok=True)
dataset = load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
dataset = dataset.cast_column("audio", Audio(decode=False))
repo_id = "davidscripka/MIT_environmental_impulse_responses"

for row in tqdm(dataset, desc="Siunčiami RIR failai"):
    hf_path = row["audio"]["path"]
    filename = os.path.basename(hf_path)
    save_path = os.path.join(rir_dir, filename)
    if not os.path.exists(save_path):
        local_file = hf_hub_download(repo_id=repo_id, filename=hf_path.split("/")[-2] + "/" + filename, repo_type="dataset")
        shutil.copy(local_file, save_path)

# 2. Foninio triukšmo (Noise) duomenys
noise_dir = "data/background_noise"
os.makedirs(noise_dir, exist_ok=True)
api = "https://api.github.com/repos/karolpiczak/ESC-50/contents/audio"
files = [f["name"] for f in requests.get(api).json() if f["name"].endswith(".wav")]
base = "https://raw.githubusercontent.com/karolpiczak/ESC-50/master/audio/"

def dl(fname):
    path = os.path.join(noise_dir, fname)
    if os.path.exists(path): return
    r = requests.get(base + fname, stream=True)
    if r.status_code == 200:
        with open(path, "wb") as f:
            for c in r.iter_content(8192):
                f.write(c)

with ThreadPoolExecutor(max_workers=16) as ex:
    list(tqdm(ex.map(dl, files), total=len(files), desc="Siunčiamas triukšmas"))

print("\nFoniniai garsai paruošti!")

  Cloning https://github.com/arcosoph/nanowakeword.git to /tmp/pip-install-ca7me2fk/nanowakeword_8413e48876124ee8a01634ea195f26b6
  Running command git clone --filter=blob:none --quiet https://github.com/arcosoph/nanowakeword.git /tmp/pip-install-ca7me2fk/nanowakeword_8413e48876124ee8a01634ea195f26b6
  Resolved https://github.com/arcosoph/nanowakeword.git to commit e89424a66f287123d47545e18a8386ac8fdf7bae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done

Diegimas baigtas. Siunčiami foniniai garsai...


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Siunčiami RIR failai: 270it [00:00, 2540.47it/s]
Siunčiamas triukšmas: 100%|██████████| 1000/1000 [00:00<00:00, 57152.45it/s]


Foniniai garsai paruošti!


In [8]:
# @title Žingsnis 2: Sugeneruojame tavo raktažodį "Asista"
import os

my_wakeword = "Asista"
output_path = "data/positive"
os.makedirs(output_path, exist_ok=True)

print(f"Generuojame garsus frazei: {my_wakeword}...")

# Sugeneruojame 1000 pavyzdžių. Jei Colab nestringa, gali padidinti iki 2000.
!nanowakeword-generate --text "{my_wakeword}" --output_dir {output_path} --count 1000

print(f"Baigta! Failai sugeneruoti į {output_path}")

Generuojame garsus frazei: Asista...
/bin/bash: line 1: nanowakeword-generate: command not found
Baigta! Failai sugeneruoti į data/positive


In [9]:
# @title Žingsnis 3: Sistemos konfigūracija
import yaml

config_dict = {
    "model_name": "asista_model",
    "output_dir": "./trained_models",
    "positive_data_path": "./data/positive",
    "negative_data_path": "./data/negative",
    "background_paths": ["./data/background_noise"],
    "rir_paths": ["./data/rir"],

    "model_type": "dnn",
    "layer_size": 32,
    "n_blocks": 3,
    "dropout_prob": 0.2,

    "steps": 20000,
    "stabilization_steps": 10000,
    "optimizer_type": "adamw",
    "learning_rate_max": 0.001,
    "lr_scheduler_type": "onecycle",
    "weight_decay": 0.01,
    "momentum": 0.9,
    "num_workers": 3,

    "feature_manifest": {
        "targets": {"t": "./trained_models/asista_model/features/positive_features.npy"},
        "negatives": {
            "n": "./trained_models/asista_model/features/negative_features.npy",
            "hn": "./trained_models/asista_model/features/hard_nagative_features.npy",
            "b": "./trained_models/asista_model/features/pure_noise_features.npy",
        },
        "targets_val": {"t_v": "./trained_models/asista_model/features/positive_features_train.npy"},
        "negatives_val": {
            "n_v": "./trained_models/asista_model/features/negative_features.npy",
            "b_v": "./trained_models/asista_model/features/pure_noise_features.npy"
        }
    },

    "batch_composition": {"t": 30, "n": 173, "hn": 20, "b": 40},

    "target_phrase": "Asista",

    "data_generation_tasks": [
        {
            "name": "Custom Negatives",
            "enabled": True,
            "output_dir": "data/negative",
            "num_samples": 50,
            "file_prefix": "neg_custom",
            "text_source": {
                "type": "from_list",
                "phrases": [
                    "Asista", "Asist", "Asisa", "Sista", "Asisto", "Ahsista"
                ],
                "repeat_each": 10
            }
        }
    ],

    "augmentation_settings": {
        "min_snr_in_db": -10.0, "max_snr_in_db": 20.0, "rir_prob": 0,
        "pitch_prob": 0.2, "min_pitch_semitones": -2.0, "max_pitch_semitones": 2.0,
        "gain_prob": 0.9, "min_gain_in_db": -6.0, "max_gain_in_db": 6.0,
        "colored_noise_prob": 0, "bg_noise_prob": 0
    },

    "augmentation_batch_size": 16,
    "feature_gen_cpu_ratio": 1.0,

    "feature_generation_manifest": {
        "pos_features": {
            "input_audio_dirs": ["./data/positive"],
            "output_filename": "positive_features.npy",
            "use_background_noise": False, "use_rir": False, "augmentation_rounds": 10
        },
        "pure_ambient_noise": {
            "input_audio_dirs": ["./data/background_noise"],
            "output_filename": "pure_noise_features.npy",
            "use_background_noise": False, "augmentation_rounds": 3,
            "augmentation_settings": {"PitchShift": 0.6, "Gain": 1.0, "RIR": 0.5}
        }
    },

    "checkpoint_averaging_top_k": 5,
    "checkpointing": {"enabled": True, "interval_steps": 1000, "limit": 2},
    "early_stopping_patience": 0,
    "main_delta": 0.0001,
    "onnx_opset_version": 17,
    "show_training_summary": True,
    "debug_mode": True,
    "ema_alpha": 0.01,

    "generate_clips": True,
    "transform_clips": True,
    "train_model": True,
    "overwrite": False
}

config_path = "./config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)

print("config.yaml failas sukurtas sėkmingai!");

config.yaml failas sukurtas sėkmingai!


In [10]:
# @title Žingsnis 4: Paleidžiame AI mokymą! 🚀
from nanowakeword.trainer import train

args_list = ['--config_path', f'{config_path}']
print("Pradedamas NanoWakeWord modelio mokymas...")

try:
    train(args_list)
    print("\n\nSVEIKINU! (✿◕‿◕✿)")
    print("Tavo 'Asista' wake word modelis sėkmingai apmokytas!")
except Exception as e:
    print(f"\nĮvyko klaida mokymo metu: {e}");

Pradedamas NanoWakeWord modelio mokymas...


  _   _               __          __   _     __          __           _ 
 | \ | |              \ \        / /  | |    \ \        / /          | |
 |  \| | __ _ _ __   __\ \  /\  / /_ _| | ____\ \  /\  / /__  _ __ __| |
 | . ` |/ _` | '_ \ / _ \ \/  \/ / _` | |/ / _ \ \/  \/ / _ \| '__/ _` |
 | |\  | (_| | | | | (_) \  /\  / (_| |   <  __/\  /\  / (_) | | | (_| |
 |_| \_|\__,_|_| |_|\___/ \/  \/ \__,_|_|\_\___| \/  \/ \___/|_|  \__,_|

INFO: Determining hardware-specific configurations...

STEP 3: Activating Synthetic Data Generation Engine

===================================================

...

Analysis complete!



                        Dataset Statistics                         
┌───────────────┬─────────────────────────────────────────────────┐
│ Parameter     │ Value                                           │
├───────────────┼─────────────────────────────────────────────────┤
│ H_pos         │ 0                                               │
│ H_neg         │ 0.018413940972222223                            │
│ H_noise_paths │ {'./data/background_noise': 1.3888888888888888} │
│ H_noise       │ 1.3888888888888888                              │
│ A_noise       │ 0.09376731533859857                             │
│ N_rir         │ 270                                             │
└───────────────┴─────────────────────────────────────────────────┘

INFO: Project assets will be saved in: /content/trained_models/asista_model

INFO: Autotuning optimal clip duration...


Įvyko klaida mokymo metu: No .wav files found for autotuning in: ./data/positive


In [11]:
# @title Žingsnis 5: Prijungiame Google Drive
from google.colab import drive

try:
    drive.mount('/content/drive')
    print("\nGoogle Drive prijungtas sėkmingai!")
except Exception as e:
    print(f"Klaida jungiantis prie Google Drive: {e}");

Klaida jungiantis prie Google Drive: Mountpoint must not already contain files


In [12]:
# @title Žingsnis 6: Nukopijuojame finalinį modelį į Drive 📂
import os
import shutil

model_name = config_dict.get("model_name", "asista_model")
output_dir = config_dict.get("output_dir", "./trained_models")

source_project_dir = os.path.join(output_dir, model_name)
drive_destination_dir = f"drive/MyDrive/nanowakeword_models/{model_name}"

print("Kopijuojami failai į Google Drive...")

if not os.path.exists(source_project_dir):
    print(f"\nKLAIDA: Nerastas aplankas '{source_project_dir}'")
else:
    if os.path.exists(drive_destination_dir):
        shutil.rmtree(drive_destination_dir)

    try:
        shutil.copytree(source_project_dir, drive_destination_dir)
        print("\n" + "="*50)
        print("✅ PUIKU! Visi failai išsaugoti tavo Google Drive.")
        print("="*50)
        print(f"\nModelio ieškok čia: '{drive_destination_dir}'")
    except Exception as e:
        print(f"\nKlaida kopijuojant: {e}")

Kopijuojami failai į Google Drive...

✅ PUIKU! Visi failai išsaugoti tavo Google Drive.

Modelio ieškok čia: 'drive/MyDrive/nanowakeword_models/asista_model'
